In [2]:
import requests
from urllib.parse import urljoin
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from bs4 import BeautifulSoup
import json
import pickle
import time
import random
import math
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict, Counter
from hazm import Normalizer, word_tokenize, stopwords_list, Stemmer
import pandas as pd

## 1. Data Collection

Crawling about 1000 Persian news articles from **Mehrnews** (https://www.mehrnews.com).  
The scraper visits paginated section pages to collect unique article URLs, then fetches each article to extract the title and body text.

In [2]:
import requests
import time
import random
import json
from urllib.parse import urljoin
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/90.0.4430.93 Safari/537.36'
}

BASE_URL = "https://www.mehrnews.com"
START_URL = "https://www.mehrnews.com/archive?tp=5"

def get_session_with_retries():
    """Create a session with very high retry tolerance."""
    session = requests.Session()
    retry_strategy = Retry(
        total=20, 
        backoff_factor=5, # 5, 10, 20, 40, ... seconds between retries
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"],
        raise_on_status=False # don't raise on 503; we handle it ourselves
    )
    adapter = HTTPAdapter(max_retries=retry_strategy)
    session.mount("http://", adapter)
    session.mount("https://", adapter)
    return session

def fetch_page(url, session, max_manual_retries=5):
    """
    Fetch a page with:
      - Built‑in session retries (from above)
      - Additional manual retries with very long waits for 503
    """
    # Polite delay before every request
    time.sleep(random.uniform(3, 6))

    for attempt in range(max_manual_retries):
        try:
            response = session.get(url, headers=HEADERS, timeout=30)
            # If we get a 503, the session's retry may have already been exhausted,
            # but we catch it here and retry manually.
            if response.status_code == 503:
                wait = (2 ** (attempt + 4)) + random.uniform(0, 5)  # 16, 32, 64, 128, 256 sec
                print(f"  503 on {url}, waiting {wait:.1f}s before manual retry {attempt+1}/{max_manual_retries}...")
                time.sleep(wait)
                continue
            response.raise_for_status()
            return response
        except requests.exceptions.HTTPError as e:
            if response.status_code == 503 and attempt < max_manual_retries - 1:
                wait = (2 ** (attempt + 4)) + random.uniform(0, 5)
                print(f"  503 (HTTPError) on {url}, waiting {wait:.1f}s (retry {attempt+1}/{max_manual_retries})...")
                time.sleep(wait)
                continue
            print(f"HTTP error {response.status_code} on {url}: {e}")
            return None
        except (requests.exceptions.ConnectionError, requests.exceptions.Timeout) as e:
            if attempt < max_manual_retries - 1:
                wait = (2 ** attempt) * 10 + random.uniform(0, 5)
                print(f"  Connection/Timeout error on {url}, waiting {wait:.1f}s (retry {attempt+1}/{max_manual_retries})...")
                time.sleep(wait)
                continue
            print(f"Error fetching {url}: {e}")
            return None
        except Exception as e:
            print(f"Unexpected error fetching {url}: {e}")
            return None

    return None

def get_article_text(article_url, session):
    """Extract full text from an article page."""
    resp = fetch_page(article_url, session)
    if not resp:
        return ""
    try:
        soup = BeautifulSoup(resp.text, 'lxml')
        article_div = (soup.find('div', class_='item-text') or
                       soup.find('div', class_='news-body') or
                       soup.find('div', id='item-body') or
                       soup.find('div', itemprop='articleBody'))
        if article_div:
            return article_div.get_text(separator="\n", strip=True)
        body_div = soup.find('div', class_='item-body')
        if body_div:
            ps = body_div.find_all('p')
            return "\n".join(p.get_text(strip=True) for p in ps)
        return ""
    except Exception as e:
        print(f"Error parsing article {article_url}: {e}")
        return ""

def get_next_page_url(soup, current_url):
    """Extract the URL of the next page from pagination."""
    next_link = soup.find('a', rel='next')
    if next_link and next_link.get('href'):
        return urljoin(BASE_URL, next_link['href'])

    pagination = soup.find('ul', class_='pagination')
    if pagination:
        active_item = pagination.find('li', class_='active')
        if active_item:
            next_item = active_item.find_next_sibling('li')
            if next_item:
                link = next_item.find('a')
                if link and link.get('href'):
                    return urljoin(BASE_URL, link['href'])
    return None

def crawl_archive(max_articles=1100):
    session = get_session_with_retries()
    articles = []
    current_url = START_URL
    page_num = 1

    while len(articles) < max_articles and current_url:
        print(f"\nFetching page {page_num}: {current_url}")
        response = fetch_page(current_url, session)
        if not response:
            print("Failed to fetch page after all retries. Stopping.")
            break

        soup = BeautifulSoup(response.text, 'lxml')
        news_items = soup.select('li.news')
        if not news_items:
            print("No news items found on this page. Stopping pagination.")
            break

        print(f"Found {len(news_items)} articles on page {page_num}.")

        for item in news_items:
            if len(articles) >= max_articles:
                break

            a_tag = item.find('a', href=True)
            title_tag = item.find('h3') or item.find('h4')
            if not a_tag or not title_tag:
                continue

            title = title_tag.get_text(strip=True)
            link = urljoin(BASE_URL, a_tag['href'])
            print(f"  [{len(articles)+1}] {title[:50]}...")

            text = get_article_text(link, session)
            articles.append({
                "title": title,
                "link": link,
                "text": text
            })

            # Longer delay between articles to reduce load
            time.sleep(random.uniform(1, 3))

        # Find next page URL
        next_url = get_next_page_url(soup, current_url)
        if next_url and next_url != current_url:
            current_url = next_url
            page_num += 1
            # Extra long pause between pages
            time.sleep(random.uniform(5, 10))
        else:
            print("No next page found. Crawling finished.")
            break

    return articles


all_articles = crawl_archive(max_articles=1100)

documents = []
for idx, art in enumerate(all_articles, 1):
    documents.append({
        "doc_id": idx,
        "title": art["title"],
        "text": art["text"],
        "url": art["link"]
    })

output_file = "data.json"
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(documents, f, ensure_ascii=False, indent=2)

print(f"\nSaved {len(documents)} articles to {output_file}")


Fetching page 1: https://www.mehrnews.com/archive?tp=5
Found 29 articles on page 1.
  [1] حضور وزیر علوم در همایش بین‌المللی فناوری‌های نوین...
  [2] تراشه کمتر از یک نانومتر در جهان ساخته شد...
  [3] توسعه دیتا سنترهای هوش مصنوعی قیمت مک بوک وآی پد ر...
  [4] امروز آخرین فرصت ثبت نام در پذیرش کاردانی به کارشن...
  [5] فراخوان کسب دانش فنی ساخت یک قطعه مهم در صنایع حسا...
  [6] آیین‌نامه جامع مدیریت دانشگاه‌ها به شورای عالی انق...
  [7] حضور و سخنرانی وزیر علوم در زینبیه استانبول همزمان...
  [8] آنتروپیک شکاف در سیستم‌های دولت آمریکا را شناسایی ...
  [9] ابررایانه چینی از رقبای آمریکایی جلو زد...
  [10] حمایت مرکز همکاری‌های علمی وزارت علوم از ابتکارات ...
  [11] تغییر برنامه زمانی امتحانات دانشجویان دانشگاه صنعت...
  [12] ظرفیت ۶۸هزار نفری کاردانی به کارشناسی؛ کامپیوتر و ...
  [13] مراسم تودیع و معارفه رئیس دانشگاه هنر برگزار شد...
  [14] خوابگاه‌های علوم پزشکی تهران میزبان مهمانان مراسم ...
  [15] پاسخ رئیس بنیاد ملی نخبگان به ابهامات جذب نخبگان د...
  [16] راکت لب ۱۶ ساعته ماموریت 

## 2. Text Preprocessing

Using the **Hazm** library we:
- Normalize Unicode characters (`Normalizer`)
- Tokenize (`word_tokenize`)
- Remove stop words (Hazm’s default list)
- Stem each token (`Stemmer`)

The preprocessed text is stored as a list of stemmed tokens (`tokens` field).

In [4]:
# Load raw data
with open('data.json', 'r', encoding='utf-8') as f:
    raw_docs = json.load(f)

# Initialize Hazm tools
normalizer = Normalizer()
stop_words = set(stopwords_list())
stemmer = Stemmer()

# Preprocess each document's text
for doc in raw_docs:
    text = doc['text']
    # 1. Normalize
    normalized = normalizer.normalize(text)
    # 2. Tokenize
    tokens = word_tokenize(normalized)
    # 3. Remove stop words
    tokens = [tok for tok in tokens if tok not in stop_words]
    # 4. Stem
    stemmed_tokens = [stemmer.stem(tok) for tok in tokens]
    doc['tokens'] = stemmed_tokens

print(f"Preprocessed {len(raw_docs)} documents.")
with open('preprocessed_data.json', 'w', encoding='utf-8') as f:
    json.dump(raw_docs, f, ensure_ascii=False, indent=2)
print("preprocessed_data.json saved.")

Preprocessed 1100 documents.
preprocessed_data.json saved.


## 3. Inverted Index

Build an inverted index `{term: {doc_id: term_frequency}}` using the stemmed tokens.

In [5]:
inverted_index = defaultdict(dict)  # term -> {doc_id: tf}

for doc in raw_docs:
    doc_id = doc['doc_id']
    term_counts = Counter(doc['tokens'])
    for term, count in term_counts.items():
        inverted_index[term][doc_id] = count

print(f"Inverted index built with {len(inverted_index)} unique terms.")
with open('inverted_index.pkl', 'wb') as f:
    pickle.dump(dict(inverted_index), f)
print("inverted_index.pkl saved.")

Inverted index built with 16143 unique terms.
inverted_index.pkl saved.


## 4. TF‑IDF Vector Space Model

We use the formula:  
`weight(term, doc) = tf(term, doc) * log10(N / df(term))`  
where N = total number of documents, df(term) = document frequency.  
Query vector uses the same idf values and raw query term frequencies.  
Retrieval is based on cosine similarity.  

We precompute document norms and then evaluate queries efficiently via the inverted index.

In [6]:
# load inverted index
with open("./inverted_index.pkl", "rb") as index_file:
    inverted_index = pickle.load(index_file)

N = len(raw_docs)
doc_norms = {} # doc_id -> L2 norm
term_idf = {} # term -> log10(N/df)

# Compute df for each term
for term, postings in inverted_index.items():
    df = len(postings)
    idf = math.log10(N / df) if df > 0 else 0.0
    term_idf[term] = idf

# Compute doc norms using the TF‑IDF weights
for doc in raw_docs:
    doc_id = doc['doc_id']
    sum_sq = 0.0
    for term, tf in Counter(doc['tokens']).items():
        idf = term_idf.get(term, 0.0)
        weight = tf * idf
        sum_sq += weight ** 2
    doc_norms[doc_id] = math.sqrt(sum_sq) if sum_sq > 0 else 1e-10  # avoid zero division

print(f"Precomputed idf for {len(term_idf)} terms and norms for {len(doc_norms)} documents.")

Precomputed idf for 16143 terms and norms for 1100 documents.
